# Construcción del dataset ML de `mag_14` con tres esquemas temporales

Se construyen tres datasets para comparar explícitamente tres situaciones:

1. **Ayer → mañana:** variables de referencia en `T-1` y target `mag_14` en `T+1`.
2. **Ayer → pasado mañana:** variables de referencia en `T-1` y target `mag_14` en `T+2`.
3. **Hoy → mañana:** variables de referencia en `T` y target `mag_14` en `T+1`.

La meteorología se utiliza siempre como **último día cerrado**, es decir, `T-1`.

La definición de variables sigue la especificación:
- meteorología: temperatura, humedad, precipitación, presión, viento y dirección/componentes;
- calendario del día objetivo;
- estación: tipo, latitud, longitud y altitud;
- histórico del contaminante objetivo: lags 1, 2 y 7, medias móviles 3 y 7;
- histórico de otros contaminantes: lags 1, 2 y 7 y medias móviles 3 y 7;
- target: concentración de `mag_14` en el horizonte correspondiente.


In [11]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "src" / "database" / "TFM.duckdb").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("No se ha encontrado la raíz del proyecto.")
    ROOT = ROOT.parent

DB_PATH = ROOT / "src" / "database" / "TFM.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

TARGET_MAG = 14
POLLUTANTS = {
    8: "no2",
    9: "o3",
    10: "pm10",
    14: "pm25",
}
HIST_LAGS = [1, 2, 7]
ROLL_WINDOWS = [3, 7]

print(f"Raíz: {ROOT}")
print(f"Base de datos: {DB_PATH}")


Raíz: c:\Users\dasab\Desktop\MASTER\TFM
Base de datos: c:\Users\dasab\Desktop\MASTER\TFM\src\database\TFM.duckdb


## 1. Cargar calidad del aire

In [12]:
# Solo estaciones que miden el contaminante objetivo.
target_stations = con.execute("""
    SELECT DISTINCT estacion
    FROM enriched.calidad_aire_final
    WHERE magnitud = 14
    ORDER BY estacion
""").df()["estacion"].tolist()

air = con.execute("""
    SELECT
        estacion,
        fecha,
        magnitud,
        uom_value
    FROM enriched.calidad_aire_final
    WHERE magnitud IN (8, 9, 10, 14)
      AND estacion IN (
          SELECT DISTINCT estacion
          FROM enriched.calidad_aire_final
          WHERE magnitud = 14
      )
    ORDER BY estacion, fecha, magnitud
""").df()

air["fecha"] = pd.to_datetime(air["fecha"])

air = (
    air.pivot_table(
        index=["estacion", "fecha"],
        columns="magnitud",
        values="uom_value",
        aggfunc="first"
    )
    .reset_index()
    .rename(columns={
        8: "no2",
        9: "o3",
        10: "pm10",
        14: "pm25",
    })
)

for col in ["no2", "o3", "pm10", "pm25"]:
    if col not in air.columns:
        air[col] = np.nan

air = air.sort_values(["estacion", "fecha"]).reset_index(drop=True)

print("Estaciones objetivo:", len(target_stations))
print("Shape aire:", air.shape)


Estaciones objetivo: 13
Shape aire: (23751, 6)


## 2. Histórico de contaminantes

In [13]:
# Lags 1, 2 y 7 días para el objetivo y los otros contaminantes.
for col in ["pm25", "no2", "o3", "pm10"]:
    for lag in HIST_LAGS:
        air[f"{col}_lag_{lag}"] = (
            air.groupby("estacion")[col].shift(lag)
        )

# Medias móviles calculadas exclusivamente sobre el pasado:
# primero se desplaza un día y después se calcula la ventana.
for col in ["pm25", "no2", "o3", "pm10"]:
    past = air.groupby("estacion")[col].shift(1)
    for window in ROLL_WINDOWS:
        air[f"{col}_mean_{window}"] = (
            past.groupby(air["estacion"]).transform(
                lambda x: x.rolling(window, min_periods=window).mean()
            )
        )


## 3. Metadatos de estación

In [14]:
# La dimensión de estaciones contiene las coordenadas y atributos estáticos.
# Se resuelven nombres habituales para evitar acoplar el notebook a una única
# denominación de columna.

schema = con.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'enriched'
      AND table_name = 'dim_estaciones_aire'
""").df()["column_name"].tolist()

def resolve_column(candidates, required=True):
    lower = {c.lower(): c for c in schema}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    if required:
        raise KeyError(
            f"No se encontró ninguna columna entre {candidates}. "
            f"Columnas disponibles: {schema}"
        )
    return None

col_station = resolve_column(["ESTACION", "estacion", "estacion_id"])
col_lat = resolve_column(["LATITUD_G", "latitude", "lat"])
col_lon = resolve_column(["LONGITUD_G", "longitude", "lon"])
col_alt = resolve_column(["ALTITUD", "altitude", "elevacion", "elevation"])
col_type = resolve_column([
    "tipo_estacion", "tipo", "tipo_de_estacion",
    "tipo_estación", "tipo_de_estación"
])

station_sql = f"""
    SELECT
        {col_station} AS estacion,
        {col_type} AS tipo_estacion,
        {col_lat} AS latitud,
        {col_lon} AS longitud,
        {col_alt} AS altitud
    FROM enriched.dim_estaciones_aire
"""

station_meta = con.execute(station_sql).df()
station_meta["estacion"] = station_meta["estacion"].astype(str)

air["estacion"] = air["estacion"].astype(str)
air = air.merge(station_meta, on="estacion", how="left")

print(station_meta.head())


  estacion tipo_estacion   latitud  longitud  altitud
0        8       TRAFICO  40.42167  -3.68222      672
1       16         FONDO  40.44000  -3.63917      698
2       17         FONDO  40.34694  -3.70500      593
3       18         FONDO  40.39472  -3.73194      625
4       24         FONDO  40.42000  -3.74917      645


## 4. Meteorología: estación más cercana y día `T-1`

In [15]:
# Estación meteorológica más cercana y válida para cada estación de aire.
mapping = con.execute("""
    WITH estaciones_aire AS (
        SELECT DISTINCT estacion
        FROM enriched.calidad_aire_final
        WHERE magnitud = 14
    ),
    estaciones_meteo AS (
        SELECT DISTINCT estacion_id
        FROM enriched.meteo_final
    ),
    distancias AS (
        SELECT
            e.ESTACION AS estacion,
            k AS estacion_meteo,
            CAST(
                json_extract(
                    e.distancias_meteo,
                    '$."' || k || '"'
                ) AS DOUBLE
            ) AS distancia_meteo
        FROM enriched.dim_estaciones_aire e
        CROSS JOIN LATERAL unnest(json_keys(e.distancias_meteo)) AS t(k)
        WHERE e.ESTACION IN (SELECT estacion FROM estaciones_aire)
          AND k IN (SELECT estacion_id FROM estaciones_meteo)
    ),
    ranking AS (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY estacion
                   ORDER BY distancia_meteo
               ) AS rn
        FROM distancias
    )
    SELECT estacion, estacion_meteo, distancia_meteo
    FROM ranking
    WHERE rn = 1
""").df()

mapping["estacion"] = mapping["estacion"].astype(str)

meteo = con.execute("""
    SELECT estacion_id, fecha, variable, uom_value
    FROM enriched.meteo_final
""").df()

meteo["fecha"] = pd.to_datetime(meteo["fecha"])
meteo["estacion_id"] = meteo["estacion_id"].astype(str)

meteo = meteo.merge(
    mapping,
    left_on="estacion_id",
    right_on="estacion_meteo",
    how="inner"
)

meteo = (
    meteo.pivot_table(
        index=["estacion", "fecha"],
        columns="variable",
        values="uom_value",
        aggfunc="first"
    )
    .reset_index()
    .sort_values(["estacion", "fecha"])
)

# Nombres esperados. Si alguna variable no existe en la fuente, queda como NaN.
METEO_MAP = {
    "temperatura": ["temperatura", "temperatura_media", "temp_mean"],
    "humedad": ["humedad", "humedad_media", "humedad_relativa"],
    "precipitacion": ["precipitacion", "precipitación"],
    "presion": ["presion", "presión", "presion_atmosferica", "presion_media"],
    "viento_velocidad": ["viento_velocidad", "velocidad_viento", "wind_speed"],
    "viento_direccion": ["viento_direccion", "direccion_viento", "wind_direction"],
}

meteo_std = pd.DataFrame({
    "estacion": meteo["estacion"],
    "fecha": meteo["fecha"],
})

for std_name, candidates in METEO_MAP.items():
    found = next((c for c in candidates if c in meteo.columns), None)
    meteo_std[std_name] = meteo[found] if found else np.nan

# La meteorología asociada a cada observación de modelado es la del día anterior.
meteo_std["fecha_referencia_meteo"] = meteo_std["fecha"] + pd.Timedelta(days=1)

meteo_tminus1 = meteo_std.drop(columns=["fecha"]).rename(
    columns={"fecha_referencia_meteo": "fecha"}
)

# Dirección del viento: componentes circulares.
angle = np.deg2rad(meteo_tminus1["viento_direccion"])
meteo_tminus1["viento_dir_sin"] = np.sin(angle)
meteo_tminus1["viento_dir_cos"] = np.cos(angle)
meteo_tminus1 = meteo_tminus1.drop(columns=["viento_direccion"])

print("Variables meteorológicas alineadas a T-1.")


Variables meteorológicas alineadas a T-1.


## 5. Calendario del día objetivo

In [16]:
def add_target_calendar(df, target_date_col):
    out = df.copy()
    d = pd.to_datetime(out[target_date_col])

    out["laborable_target"] = d.dt.dayofweek < 5
    out["mes_sin"] = np.sin(2 * np.pi * d.dt.month / 12)
    out["mes_cos"] = np.cos(2 * np.pi * d.dt.month / 12)

    return out


## 6. Construcción de los tres casos

In [17]:
# ============================================================
# CONSTRUCCIÓN DE LOS 3 ESCENARIOS TEMPORALES
# ============================================================
#
# A) ayer -> mañana:
#    variables de contaminación: T-1
#    meteorología:               T-1
#    calendario:                  T+1
#    target:                      T+1
#
# B) ayer -> pasado mañana:
#    variables de contaminación: T-1
#    meteorología:               T-1
#    calendario:                  T+2
#    target:                      T+2
#
# C) hoy -> mañana:
#    variables de contaminación: T
#    meteorología:               T-1
#    calendario:                  T+1
#    target:                      T+1
# ============================================================

# ------------------------------------------------------------
# BASE T: datos de contaminación del día T
# ------------------------------------------------------------

base_t = air.copy()

# Meteorología siempre del día anterior (T-1).
base_t = base_t.merge(
    meteo_tminus1,
    on=["estacion", "fecha"],
    how="left"
)

# Fechas objetivo.
base_t["target_t1_date"] = base_t["fecha"] + pd.Timedelta(days=1)
base_t["target_t2_date"] = base_t["fecha"] + pd.Timedelta(days=2)

# Targets.
targets = air[["estacion", "fecha", "pm25"]].copy()

target_t1 = targets.rename(
    columns={
        "fecha": "target_t1_date",
        "pm25": "target_t1"
    }
)

target_t2 = targets.rename(
    columns={
        "fecha": "target_t2_date",
        "pm25": "target_t2"
    }
)

base_t = base_t.merge(
    target_t1,
    on=["estacion", "target_t1_date"],
    how="left"
)

base_t = base_t.merge(
    target_t2,
    on=["estacion", "target_t2_date"],
    how="left"
)


# ------------------------------------------------------------
# BASE T-1: desplazamos TODAS las variables de contaminación
# para el escenario "ayer".
# ------------------------------------------------------------

air_tminus1 = air.copy()

air_tminus1["fecha"] = (
    air_tminus1["fecha"] + pd.Timedelta(days=1)
)

# Renombramos las columnas de contaminación para diferenciarlas
# de las correspondientes a T.
air_tminus1 = air_tminus1.rename(
    columns={
        col: f"{col}_tminus1"
        for col in air_tminus1.columns
        if col not in ["estacion", "fecha"]
    }
)

# Nos quedamos solo con las variables históricas/predictoras
# que existen en air y que queremos trasladar a T-1.
base_ayer = base_t.merge(
    air_tminus1,
    on=["estacion", "fecha"],
    how="inner"
)


# ------------------------------------------------------------
# CASO A: AYER -> MAÑANA
# ------------------------------------------------------------

ayer_manana = base_ayer.copy()

ayer_manana["caso"] = "ayer_manana"

# La referencia real del escenario es T-1.
ayer_manana["fecha_referencia"] = (
    ayer_manana["fecha"] - pd.Timedelta(days=1)
)

ayer_manana["fecha_target"] = ayer_manana["target_t1_date"]
ayer_manana["target"] = ayer_manana["target_t1"]

# Calendario según el día objetivo T+1.
ayer_manana = add_target_calendar(
    ayer_manana,
    "fecha_target"
)


# ------------------------------------------------------------
# CASO B: AYER -> PASADO MAÑANA
# ------------------------------------------------------------

ayer_pasado = base_ayer.copy()

ayer_pasado["caso"] = "ayer_pasado_manana"

# La referencia sigue siendo T-1.
ayer_pasado["fecha_referencia"] = (
    ayer_pasado["fecha"] - pd.Timedelta(days=1)
)

ayer_pasado["fecha_target"] = ayer_pasado["target_t2_date"]
ayer_pasado["target"] = ayer_pasado["target_t2"]

# Calendario según el día objetivo T+2.
ayer_pasado = add_target_calendar(
    ayer_pasado,
    "fecha_target"
)


# ------------------------------------------------------------
# CASO C: HOY -> MAÑANA
# ------------------------------------------------------------

hoy_manana = base_t.copy()

hoy_manana["caso"] = "hoy_manana"

# La referencia real es T.
hoy_manana["fecha_referencia"] = hoy_manana["fecha"]

hoy_manana["fecha_target"] = hoy_manana["target_t1_date"]
hoy_manana["target"] = hoy_manana["target_t1"]

# Calendario según el día objetivo T+1.
hoy_manana = add_target_calendar(
    hoy_manana,
    "fecha_target"
)


# ------------------------------------------------------------
# DATASETS FINALES
# ------------------------------------------------------------

datasets = {
    "ayer_manana": ayer_manana,
    "ayer_pasado_manana": ayer_pasado,
    "hoy_manana": hoy_manana,
}

print("Escenarios construidos:")
for name, df in datasets.items():
    print(
        f"  {name}: "
        f"{len(df)} filas | "
        f"referencia {df['fecha_referencia'].min()} -> {df['fecha_referencia'].max()} | "
        f"target {df['fecha_target'].min()} -> {df['fecha_target'].max()}"
    )


Escenarios construidos:
  ayer_manana: 23738 filas | referencia 2020-01-01 00:00:00 -> 2024-12-30 00:00:00 | target 2020-01-03 00:00:00 -> 2025-01-01 00:00:00
  ayer_pasado_manana: 23738 filas | referencia 2020-01-01 00:00:00 -> 2024-12-30 00:00:00 | target 2020-01-04 00:00:00 -> 2025-01-02 00:00:00
  hoy_manana: 23751 filas | referencia 2020-01-01 00:00:00 -> 2024-12-31 00:00:00 | target 2020-01-02 00:00:00 -> 2025-01-01 00:00:00


## 7. Selección final de variables

In [18]:
AIR_FEATURES = []

for col in ["pm25", "no2", "o3", "pm10"]:
    AIR_FEATURES += [f"{col}_lag_{lag}" for lag in HIST_LAGS]
    AIR_FEATURES += [f"{col}_mean_{w}" for w in ROLL_WINDOWS]

METEO_FEATURES = [
    "temperatura",
    "humedad",
    "precipitacion",
    "presion",
    "viento_velocidad",
    "viento_dir_sin",
    "viento_dir_cos",
]

STATION_FEATURES = [
    "tipo_estacion",
    "latitud",
    "longitud",
    "altitud",
]

CALENDAR_FEATURES = [
    "laborable_target",
    "mes_sin",
    "mes_cos",
]

IDENTIFIERS = [
    "estacion",
    "fecha_referencia",
    "fecha_target",
    "caso",
]

FINAL_FEATURES = IDENTIFIERS + STATION_FEATURES + AIR_FEATURES + METEO_FEATURES + CALENDAR_FEATURES + ["target"]

for name, df in datasets.items():
    missing_cols = [c for c in FINAL_FEATURES if c not in df.columns]
    if missing_cols:
        raise KeyError(f"{name}: faltan columnas {missing_cols}")
    datasets[name] = df[FINAL_FEATURES].copy()

print("Features:", len(FINAL_FEATURES) - len(IDENTIFIERS) - 1)
print("Total columnas:", len(FINAL_FEATURES))


Features: 34
Total columnas: 39


## 8. Validaciones temporales

In [19]:
# ============================================================
# COMPROBACIÓN DE LAS RELACIONES TEMPORALES
# ============================================================

# Ayer -> mañana:
# T-1 -> T+1 = 2 días
assert (
    (
        datasets["ayer_manana"]["fecha_target"]
        - datasets["ayer_manana"]["fecha_referencia"]
    ).dt.days == 2
).all()

# Ayer -> pasado mañana:
# T-1 -> T+2 = 3 días
assert (
    (
        datasets["ayer_pasado_manana"]["fecha_target"]
        - datasets["ayer_pasado_manana"]["fecha_referencia"]
    ).dt.days == 3
).all()

# Hoy -> mañana:
# T -> T+1 = 1 día
assert (
    (
        datasets["hoy_manana"]["fecha_target"]
        - datasets["hoy_manana"]["fecha_referencia"]
    ).dt.days == 1
).all()

# Comprobamos también el identificador de cada escenario
assert datasets["ayer_manana"]["caso"].eq("ayer_manana").all()
assert datasets["ayer_pasado_manana"]["caso"].eq("ayer_pasado_manana").all()
assert datasets["hoy_manana"]["caso"].eq("hoy_manana").all()

print("✅ Relaciones temporales verificadas:")
print("  ayer -> mañana:          T-1 -> T+1 = 2 días")
print("  ayer -> pasado mañana:   T-1 -> T+2 = 3 días")
print("  hoy -> mañana:           T   -> T+1 = 1 día")

print()
print("Filas:")
print("  ayer -> mañana:", len(datasets["ayer_manana"]))
print("  ayer -> pasado mañana:", len(datasets["ayer_pasado_manana"]))
print("  hoy -> mañana:", len(datasets["hoy_manana"]))

✅ Relaciones temporales verificadas:
  ayer -> mañana:          T-1 -> T+1 = 2 días
  ayer -> pasado mañana:   T-1 -> T+2 = 3 días
  hoy -> mañana:           T   -> T+1 = 1 día

Filas:
  ayer -> mañana: 23738
  ayer -> pasado mañana: 23738
  hoy -> mañana: 23751


## 9. Eliminar filas sin información mínima

In [20]:
# Para entrenar cada escenario se exige:
# - histórico mínimo del contaminante objetivo;
# - target disponible.
#
# Las variables meteorológicas estructuralmente no disponibles se conservan
# como NaN para no fabricar información.

required = [
    "pm25_lag_1",
    "pm25_lag_2",
    "pm25_lag_7",
    "pm25_mean_3",
    "pm25_mean_7",
    "target",
]

for name in list(datasets):
    before = len(datasets[name])
    datasets[name] = datasets[name].dropna(subset=required).copy()
    after = len(datasets[name])
    print(f"{name}: {before} -> {after} filas")


ayer_manana: 23738 -> 23647 filas
ayer_pasado_manana: 23738 -> 23634 filas
hoy_manana: 23751 -> 23647 filas


## 10. Guardar los tres datasets

## 11. Resumen

In [21]:
for name, df in datasets.items():
    print("\n", name)
    print("  filas:", len(df))
    print("  estaciones:", df["estacion"].nunique())
    print("  referencia:", df["fecha_referencia"].min(), "->", df["fecha_referencia"].max())
    print("  target:", df["fecha_target"].min(), "->", df["fecha_target"].max())
    print("  columnas:", len(df.columns))

con.close()
print("\nConexión DuckDB cerrada.")



 ayer_manana
  filas: 23647
  estaciones: 13
  referencia: 2020-01-07 00:00:00 -> 2024-12-29 00:00:00
  target: 2020-01-09 00:00:00 -> 2024-12-31 00:00:00
  columnas: 39

 ayer_pasado_manana
  filas: 23634
  estaciones: 13
  referencia: 2020-01-07 00:00:00 -> 2024-12-28 00:00:00
  target: 2020-01-10 00:00:00 -> 2024-12-31 00:00:00
  columnas: 39

 hoy_manana
  filas: 23647
  estaciones: 13
  referencia: 2020-01-08 00:00:00 -> 2024-12-30 00:00:00
  target: 2020-01-09 00:00:00 -> 2024-12-31 00:00:00
  columnas: 39

Conexión DuckDB cerrada.


In [23]:
from pathlib import Path

# Carpeta de salida del proyecto
OUTPUT_DIR = ROOT / "data" / "modeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Rutas de los tres datasets
paths = {
    "ayer_manana": OUTPUT_DIR / "dataset_mag14_ayer_a_manana.parquet",
    "ayer_pasado_manana": OUTPUT_DIR / "dataset_mag14_ayer_a_pasado_manana.parquet",
    "hoy_manana": OUTPUT_DIR / "dataset_mag14_hoy_a_manana.parquet",
}

# Asegurar que tipo_estacion sea numérico para XGBoost
mapa_tipo_estacion = {
    "tráfico": 1,
    "trafico": 1,
    "fondo": 0,
}

for nombre in ["ayer_manana", "ayer_pasado_manana", "hoy_manana"]:
    df = datasets[nombre].copy()

    if "tipo_estacion" in df.columns:
        df["tipo_estacion"] = (
            df["tipo_estacion"]
            .astype("string")
            .str.strip()
            .str.lower()
            .map(mapa_tipo_estacion)
            .astype("Int8")
        )

    datasets[nombre] = df

# Guardar los tres datasets
for nombre, path in paths.items():
    datasets[nombre].to_parquet(path, index=False)
    print(f"{nombre}: guardado correctamente -> {path}")

ayer_manana: guardado correctamente -> c:\Users\dasab\Desktop\MASTER\TFM\data\modeling\dataset_mag14_ayer_a_manana.parquet
ayer_pasado_manana: guardado correctamente -> c:\Users\dasab\Desktop\MASTER\TFM\data\modeling\dataset_mag14_ayer_a_pasado_manana.parquet
hoy_manana: guardado correctamente -> c:\Users\dasab\Desktop\MASTER\TFM\data\modeling\dataset_mag14_hoy_a_manana.parquet
